# 04 — Product & Margin Analysis

**Olist Customer Intelligence Platform**

Which product categories are healthy, and which quietly generate dissatisfaction?
We score every category on **revenue**, **volume**, **satisfaction** (avg review),
and **cancellation rate**, then plot them so the problem categories fall out
visually.

> **Data honesty:** Olist has no refund/return field, so we do *not* claim a
> "return rate." We proxy product problems with **review score** (satisfaction)
> and **cancellation rate** (orders abandoned before delivery). Both are real,
> measurable signals; neither is literally a return.

In [1]:
import duckdb
import pandas as pd
import plotly.express as px
from pathlib import Path

DB_PATH = None
for candidate in (Path("../data/olist.duckdb"), Path("data/olist.duckdb")):
    if candidate.exists():
        DB_PATH = candidate.resolve()
        break
if DB_PATH is None:
    raise FileNotFoundError("olist.duckdb not found — run scripts/load_raw.py and `dbt build`.")

con = duckdb.connect(str(DB_PATH), read_only=True)


def q(sql: str) -> pd.DataFrame:
    """Run SQL against the Olist DuckDB and return a DataFrame."""
    return con.execute(sql).df()


print(f"Connected (read-only) to {DB_PATH.name}")

Connected (read-only) to olist.duckdb


## Step 1 — Build the category scorecard (your SQL)

Return one row per category, assigned to a variable named `scorecard`, with:

| column | meaning |
|---|---|
| `product_category` | the English category name |
| `revenue` | delivered item revenue (`sum(price)`, delivered orders only) |
| `delivered_orders` | distinct delivered orders containing the category |
| `avg_review` | average review score at **order grain** (not item-weighted) |
| `cancel_rate` | % of *all* orders touching the category that were canceled |

**Suggested CTEs:**
1. `order_categories` — `select distinct order_id, order_status, product_category`
   from `mart_orders` → `stg_order_items` → `int_products_translated`.
   (DISTINCT collapses multi-item orders so an order counts once per category.)
2. `cat_orders` — per category: `count(distinct order_id)` and
   `count(distinct order_id) filter (where order_status='canceled')` →`cancel_rate`.
3. `cat_revenue` — delivered only: `sum(i.price)` revenue + distinct delivered orders.
4. `cat_review` — from the DISTINCT (order, category) rows (delivered),
   `avg(avg_review_score)` so each order's review counts once, not once per item.
5. Join them on `product_category`.

⚠️ Note why step 4 uses the DISTINCT set: if you average `avg_review_score` over
the raw item join, a 3-item order's review counts 3× (the item-weighting bug we
flagged in Q9).

In [2]:
scorecard = q("""
with order_categories as (
    select distinct
        o.order_id,
        o.order_status,
        p.product_category,
        o.avg_review_score
    from mart_orders o
    join stg_order_items i
        on o.order_id = i.order_id
    join int_products_translated p
        on i.product_id = p.product_id
),

cat_orders as (
    select
        product_category,
        round(100.0 * count(distinct order_id) filter (where order_status = 'canceled')
            / count(distinct order_id), 2) as cancel_rate
    from order_categories
    group by 1
),

cat_revenue as (
    select
        p.product_category,
        round(sum(i.price), 2) as revenue,
        count(distinct o.order_id) as delivered_orders
    from stg_order_items i
    join mart_orders o
        on i.order_id = o.order_id
        and o.order_status = 'delivered'
    join int_products_translated p
        on i.product_id = p.product_id
    group by 1
),

cat_review as (
    select
        product_category,
        round(avg(avg_review_score), 2) as avg_review
    from order_categories
    where order_status = 'delivered'
      and avg_review_score is not null
    group by 1
)

select
    cr.product_category,
    cr.revenue,
    cr.delivered_orders,
    rv.avg_review,
    co.cancel_rate
from cat_revenue cr
left join cat_review rv
    on cr.product_category = rv.product_category
left join cat_orders co
    on cr.product_category = co.product_category
order by cr.revenue desc
""")
scorecard.sort_values("revenue", ascending=False).head(10)

,product_category,revenue,delivered_orders,avg_review,cancel_rate
0,health_beauty,1233131.72,8647,4.23,0.41
1,watches_gifts,1166176.98,5495,4.12,0.36
2,bed_bath_table,1023434.76,9272,4.00,0.19
3,sports_leisure,954852.55,7530,4.23,0.61
4,computers_accessories,888724.61,6530,4.08,0.52
5,furniture_decor,711927.69,6307,4.06,0.37
6,housewares,615628.69,5743,4.19,0.63
7,cool_stuff,610204.10,3559,4.22,0.41
8,auto,578966.65,3810,4.15,0.62
9,toys,471286.48,3804,4.24,0.80


## Step 2 — The category scorecard chart (provided)

Revenue (y) vs satisfaction (x), bubble size = volume, color = cancellation rate.
The dashed line is the volume-weighted average satisfaction: categories that sit
**high and to the left of it** are high-revenue but below-average satisfaction —
the ones to investigate.

In [3]:
top = scorecard.nlargest(25, "revenue").copy()

avg_sat = (top["avg_review"] * top["delivered_orders"]).sum() / top["delivered_orders"].sum()

fig = px.scatter(
    top, x="avg_review", y="revenue",
    size="delivered_orders", color="cancel_rate",
    hover_name="product_category", size_max=45,
    color_continuous_scale="Reds",
    labels=dict(avg_review="Avg review score (satisfaction)",
                revenue="Delivered revenue (R$)",
                delivered_orders="Order volume", cancel_rate="Cancel %"),
    title="Category scorecard — top 25 by revenue",
)
fig.add_vline(x=avg_sat, line_dash="dash",
              annotation_text="avg satisfaction", annotation_position="top")
fig.show()

## Step 3 — Findings

**Takeaway:** _Among the top 25 categories, bed/bath (R$1.0M), watches/gifts (R$1.2M), computers/accessories, and furniture/decor all sit below the 4.14 avg satisfaction line — with office furniture the worst at 3.64. Musical instruments and small appliances carry the highest cancel rates (~1.3%) despite modest volume. If fixing one category: **bed/bath table** — highest combined revenue exposure (R$1.0M) with below-average satisfaction and meaningful order volume (9.3K orders). **Executive:** the biggest revenue categories aren't the happiest customers; quality and fulfillment fixes in home/furniture should come before chasing new category expansion._